# Анализ непараметрических тестов

В этом ДЗ вам предстоит поиграться с непараметрическими тестами: вывести и имплементировать их, а затем сравнить их мощность с классическими параметрическими тестами.

Напоминаем, что пользоваться LLM можно только для генерации графиков, и в таком случае прикладывать промпт и комментарии о ручных изменениях результата генерации.

## Контекст и постановка задачи

Мы изучаем, влияет ли фоновый шум на скорость реакции человека. Наши коллеги уже поставили эксперимент: каждый из 20 участников прошёл тест на скорость реакции, и сделал это дважды: в тихой комнате, и в комнате с фоновым шумом. Порядок испытаний (шум -> тишина или тишина -> шум) был случайным образом назначен каждому участнику. Участники не общались между собой.

Испытание состоит из серии шагов, где каждый шаг - последовательное возникновение стимулов (картинок) на экране; при виде красного круга (и только него) участник должен как можно быстрее нажать на кнопку на приборе в его руке. Красный круг появляется несколько раз за сессию. При появлении красного круга засекается время до нажатия (= время реакции).  Результат каждого испытания — медианное время реакции (в миллисекундах) за сессию. Следовательно, данные парные: каждый участник выступает своим собственным контролем.

## 1. Анализ дизайна эксперимента

**1.1** Опишите ваши данные при помощи статистической выборочной модели (т.е. как выборку из случайного вектора), укажите, за что отвечает каждая случайная величина. Какими свойствами она обладает? Будут ли случайные величины в рамках одного вектора-наблюдения независимы?



Выборка представляет собой 20 независимых одинаково распределеннных случайных векторов

Пусть $T_i$ – медианное время реакции i-го участника в тишине\
Пусть $N_i$ – медианное время реакции i-го участника в шуме

Тогда у нас вектор i-го участника $X_i = (T_i, N_i)$

---

Вероятно, в рамках одного вектора наблюдения у нас св не будут независимы, потому что как минимум индивидуальная базовая скорость реакции влияет на оба измерения


**1.2** Для чего в условиях эксперимента порядок испытаний для каждого участника назначается случайно, какую прикладную проблему это решает? В чем была бы проблема выводов из результатов эксперимента, если бы все участники получили одинаковый фиксированный порядок испытаний? Почему важно условие о том, что участники не общались между собой?



Если бы все испытуемые получули одинаковый порядок, то мы бы тестировали не влияние шума на реакцию как таковое, а влияние шума при именно такой последовательности испытаний. Тогда у нас эффект буквально был бы смещен (оценка эффекта шума могла бы быть смешана с эффектом порядка), потому что например участники могли в первый тихий этап просто приноровиться, и во втором показать себя лучше (то есть не полный эффект шума) или наоборот они просто не ожидали такого задания и были в шоке, поэтому хуже показали себя на первом этапе (с шумом), но потом взяли себя в руки. Поэтому мы распределяем порядки случайно, чтобы сгладить все эффекты

Условие о том, что участники не общались между собой, важно для независимости наблюдений. Если участники могли бы обмениваться информацией, они могли бы рассказывать друг другу о структуре задания, картинках или стратегиях прохождения теста. Тогда результат одного участника мог бы влиять на результат другого, и предположение о независимости разных векторов наблюдений стало бы сомнительным

## 2. Т-тест

**2.1** Опишите теоретический измеренный эффект шума на скорость реакции для i-го человека как выражение от ваших случайных величин.

$d_i = N_i - T_i$


Для удобства, выберите запись, в которой ожидаемое по контексту значение $d_i$ выше нуля.

**2.2**

Предположим, что измеренные нами медианы каждого участника в каждом из случаев распределены нормально с некоторыми неизвестными параметрами.

Выведите распределение статистики $Z = \frac{\overline{d}}{\sqrt{D(\overline{d})}}$ в предположении, что средние медианы каждого испытуемого не отличаются в различных условиях.

Все ли компоненты этой статистики мы можем рассчитать точно при текущих предпосылках и вводных? Если нет, то сможем ли, если будем знать дисперсии частных распределений каждой из групп?

$$d_i \sim N(0, \sigma^2) \implies \bar{d} = \frac{1}{n} \sum d_i \sim N(0, \frac{\sigma^2}{n})$$
$$Z = \frac{\bar{d}}{\sqrt{D(\bar{d})}} = \frac{\bar{d}\cdot\sqrt{n}}{\sigma} \sim N(0, \frac{\sigma^2}{n}\cdot\frac{n}{\sigma^2}) = \boxed{N(0, 1)} $$

Мы не можем посчитать $D(\bar{d})$ сейчас и в ситуации, коогда у нас будут дисперсии частныэ распределений. Мы не знаем ковариацию между T_i и N_i поэтому не сможем все расчитать (а она точно есть, потому что у нас парная статистика)

**2.3** 

Выведите распределение статистики

$T = \dfrac{ \overline{d} }{ \sqrt{D( \overline{d} )} } \cdot \sqrt{ \dfrac{D(\overline{d})}{\widehat{D(\overline{d})} } }$


где $\widehat{D(\overline{d})} = \frac{1}{n} \widehat{D(d)} = \frac{1}{n} \left( \frac{1}{n-1} \sum_{i=1}^n (d_i - \overline{d})^2 \right)$

Воспользуйтесь предыдущим пунктом, тем что отношение оценки дисперсии и истинной дисперсии распределено попроционально $\chi^2$ и независимо от среднего.


Можем ли мы рассчитать эту статистику на основе имеющихся данных? Если нет, то чего не хватает?

Является ли этот тест одновыборочным или двухвыборочным, почему?


<!-- а также теоремой Манна-Вальда (произведение a*b - почти непрерывная функция по обоим аргументам) и соотношением между различными видами сходимости. -->

$$T = Z\cdot\sqrt{\frac{D(\bar{d})}{\hat{D(\bar{d})}}} $$

Пусть $U = \frac{(n-1)s^2}{\sigma^2} \implies U \sim \chi^2_{n-1}$, тогда:
$$T = Z\cdot\sqrt{\frac{n-1}{U}} = \frac{Z}{\sqrt{U/(n-1)}} \sim t_{n-1} = \boxed{t_{19}} $$

---

Да, вот тут уже можем расчитать статистику. В прошлый раз загвоздка была в истинной дисперсии, а тут она больше не фигурируе – проблем нет

Ну вообще это одновыборочный тест, потому что все таки мы сравниваем только одну св с конкретным значением

**2.4**  

Опираясь на прикладную задачу, сформулируйте нулевую и двустороннюю альтернативную гипотезу, 


Приведите контекстуальные (по нашей задаче) интерпретации для гипотез. Для каждой гипотезы укажите одну интерпретацию, вытекающую непосредственно из постановки теста как статистической модели, и одну, которая следует и из постановки теста, и из дизайна эксперимента (каузальную).

Укажите область отвержения H0. Зависит ли она от $\alpha$, $n$, $\beta$?  

Выведите p-value как функцию от $T_{\text{obs}}$.



$H_0 : \mu = 0$, то есть среднее значение индивидуального эффекта d = N - T равно нулю: фоновый шум не оказывает причинного влияния на медианное время реакции человека\
$H_1 : \mu != 0$, то есть в генеральной совокупности средняя разность между медианным временем реакции в шуме и медианным временем реакции в тишине не равна нулю: фоновый шум в среднем оказывает причинное влияние на медианное время реакции человека

---

$$T \in R = (t_{\alpha/2;\space n-1},\space t_{1-\alpha/2;\space n-1}) $$
$$P(T \in R) = 1 - \alpha $$

Область отвержения будет напрямую зависеть от alpha и от n (альфа – допустимый уровень значимости; от n зависит распределение тк у него n-1 степеней свободы)

---

тк распределение Стьюдента симметричное:
$$p_{value} = P(|T_{n-1}| \ge |T_{obs}|) $$
$$p_{value} = 2P(T_{n-1} \ge |T_{obs}|) $$
$$p_{value} = 2(1 - F_{n-1}(|T_{obs}|))$$


## 3. Ранговый тест: Wilcoxon signed-rank test

*Намекнуть, как собрать тестовую статистику, показать её динамику аналитически, вывести аналитически MDE?*





Рассмотрим более реалистичную ситуацию: предпосылки о нормальности исходных распределений нет и в помине, асимптотическая аппроксимация T -> Z при n=20 не работает. Что будем делать? Обратимся к непараметрическим тестам.

Начнём с ранговых тестов, а именно - Wilcoxon runk-sum test.



*Идея:* по сути, у нас на руках находится выборка из разниц для каждого человека в разных условиях. В параметрических тестах мы пытались вывести ее распределение, опираясь на факты о распределении исходных наблюдений. А что, если попробовать взять более широкую нулевую гипотезу, на основе нее получить распределение статистики при верной H0, и искать противоречия? При этом, чтобы опираться только на данные у нас на руках и не зависеть от значений (и исходных распределений), можно перейти к рангам значений.



**3.0.1** *Нулевая гипотеза*
- Предположим, что в рамках одного испытуемого распределения медианных времен реакций в обеих группах одинаковые: $F_i = G_i$
  - Оставаясь в рамках одного испытуемого мы позволяем себе гетерогенных участников с разными функциями распределения между собой. Это нормально и очень желательно, поскольку нас не интересуют различия между людьми, мы не хотим быть к ним чувствительными.
  - $H_0: F_i = G_i \quad \forall i$
- Тогда $P(d_i > 0) = P(d_i < 0) = 0.5$, поскольку вероятность того, что мы возьмем два наблюдения $X_i$, $Y_i$  из распределений $F_i$, $G_i$ (в таком порядке) равна вероятности того, что мы возьмем их из распределений $G_i$, $F_i$ (в таком порядке), поскольку распределения одинаковые.
- Фактически, это знакомая вам гипотеза о стохастическом доминировании
- Альтернативно, можно рассмотреть менее требовательную гипотезу, из которой следует всё то же: медиана d равна нулю.
  - $H'_0: med(d_i) = 0 \quad \forall i$


<!--- Как можно искать ей противоречия? По-всякому, но нас интересуют такие, что будут чувствительны к разности "центров" (средних, медиан) этих распределений; устроит нас и свидетельство того, что основные массы вероятностей этих распределений находятся далеко друг от друга. -->

**3.1** Построение тестовой статистики.

Упорядочим нашу выборку разностей по модулю, от наименьшего к наибольшему. Для удобства предположим, что там нет "ничьих" (равенств наблюдений), а также нулевых разностей, и порядок можно сделать строгим. Назначим каждому наблюдению его ранг по модулю: это его номер от 1 до n в упорядоченном по модулю ряду. $R_i$ - ранг i-го наблюдения (i-го в нумерации исходной (неранжированной) выборки). 

Возьмем две статистики:

$W^- = \sum_{i: D_i < 0 } R_i$

$W^+ = \sum_{i: D_i > 0 } R_i$

**3.1.1** Чему равно $W^- + W^+$?

**3.1.2** Каковы ожидаемые значения этих статистик при верной H0?

**3.1.3** Какова вероятность получить множество $\{i: \quad D_i < 0\}$ равным любому конкретному, заранее выбранному множеству рангов? (например, {1, 15, 9, 17} или {16, 18, 5, 4, 6, 10, 14}).


**3.1.4**  Представьте статистику как сумму всех рангов, умножаемых на независимые бернуллевские величины с p=0.5. При помощи этого представления найдите дисперсии статистик $W^-$, $W^+$, раскрыв дисперсию по свойствам. Объясните, почему так можно сделать.
- Не забудьте базовые факты про суммы степеней первых n натуральных чисел. 


**3.1.5** Как точно вычислить $P(W^+ = k), P(W^+ = k)$? k - параметр, фиксированное число.
- *Подсказка:* не пытайтесь привести явную формулу, ее либо не существует, либо она слишком сложна. Опишите, как *алгоритмически* можно вычислить такую вероятность. Пункт 3.1.4 и соображения о  классической схеме вероятности могут вам помочь. Какую алгоритмическую сложность имеет ваше решение?


**3.1.6** Какое асимптотическое распределение имеют статистики $W^+$, $W^-$? Предъявите явное преобразование статистики, которое имеет стандартное нормальное распределение.
- Подсказка: в этом случае у вас не сработает стандартная ЦПТ, поскольку компоненты суммы разнораспределенные и с увеличивающейся дисперсий. (Кстати, почему?) Здесь придется поверить на слово: здесь работает другая, более сложная версия ЦПТ (Линдеберга), и условия для нее выполняются. Вам достаточно применить обычную z-нормализацию.


---


3.1.1 мы просто суммируем все ранги (учитывая что нулевых щзначений у нас нет), поэтому получим просто $\sum_{i=1}^ni $\
3.1.2 При верной H_0 у нас они (ну их матожи) должны быть обе равны , то есть в каждой половинка от $\sum_{i=1}^ni $\
3.1.3 Ну здесь по факту у нас те, что в нужном наборе должны попасть на левую сторону, а остальная часть – на правую. Обе вероятности 0.5 поэтому у нас будет просто $0.5^n = 2^{-n}$ \
3.1.4 $W^- = \sum_{i=1}^ni\cdot B_i$, где $B_i = {0, 1}; p=0.5$ \
Тогда $Var(W^-) = Var(\sum i\cdot B_i) = \sum Var(i\cdot B_i) = \sum i^2 Var(B_i) = \sum i^2 p(1-p) = \sum_{i=1}^n i^2\cdot \frac{1}{4} = \frac{1}{4} \sum_{i=1}^n i^2  =\boxed{ \frac{n(n+1)(2n+1)}{24}} \quad$. Можем так сделать, потому что известно, что B_i независимы \
С $W^+$ все то же самое, или можно просто вычесть из всего полученный $W^-$

3.1.5 Ну тут наверное можем просто поделить число удовлетворяющих условиям вариантов на общее число вариантов. Общее число мы уже знаем – $2^n$, а число удовлетворяющих нас раскладов можно найти через дп по суммам:

пусть dp[s] = сколько способов набрать сумму s из уже рассмотренных рангов, тогда начинаем с dp[0] = 1 и так идем до конца, постоянно обновляя. Тогда у нас $P(W^+ = k) = \frac{dp[k]}{2^n} $\
По сложности вероятно будет что то вида O(n^3), потому что мы пробегаем для нужного значения еще n раз, а чтобы дойти до нужного значения может потребоваться $\frac{n(n+1)}{2} $ ходов (максимальный ранг)

3.1.6 Мы уже находили дисперсию, теперь найдем матож:\
$E(W^-) = E(\sum i\cdot B_i) = \sum E(i \cdot B_i) = \sum i \cdot E(B_i) = \sum ip = \frac{1}{2} \sum i = \frac{1}{2} \frac{n(n+1)}{2} = \boxed{\frac{n(n+1)}{4}} $

Тогда $W^- \approx N(\frac{n(n+1)}{4}, \frac{n(n+1)(2n+1)}{24}) \implies \frac{W^- - \frac{n(n+1)}{4}}{\sqrt{\frac{n(n+1)(2n+1)}{24}}} \Rightarrow N(0, 1) $

Стандартная ЦПТ не рботает, потому что у нас СВ не одинаково распределены. У нас одинаковые B_i но мы умножаем их на разные числа, то есть будут разные распределене и дисперсия

---

**3.2** Критерий принятия решения

Сформулируйте статистическую двустороннюю альтернативную гипотезу о стохастическом доминировании. Для нее сформулируйте критерий отвержения $H_0$ через значения статистик $W^+$ и $W^-$ для данного $\alpha$, а также формулу p-value; помните про его определение во всех шагах этой задачи.

**3.2.1** Сформулируйте указанное для асимптотической версии теста.

**3.2.2** Сформулируйте указанное для точной версии теста.
- Здесь можно описать идею текстом, если аккуратно записать формулами дается тяжело
- Разумнее сначала сделать критерий отвержения через p-value, а затем описать логику критерия через критические значения

**3.3.3** Какова интерпретация альтернативной гипотезы с точки зрения постановки теста; с точки зрения постановки теста и дизайна эксперимента совместно (каузальная интерпретация)?



H_1: присутствует стохастическое доминиирование в любую из сторон

Для асимптотической версии теста мы просто берем нашу Z статистику (из пункта 3.1.6) и используем стандартную практику: отвергаем, если $|Z_{obs}| \ge z_{1-\alpha/2};\quad p_{value} = P(|Z| \ge |Z_{obs}|) = 2(1 - \Phi(|Z_{obs}|)) $

Для точной версии теста нам будет удобно ввести статистику $M_{obs} = min(W_{obs}^-, W_{obs}^+) $. Дальше мы просто будем делать $p_{value} = P(M \le M_{obs}) $, то есть будем смотреть вероятность получить такое же или более экстремальное значение. Тут мы учитываем двусторонность гипотезы, потому что если у нас одно значение экстремально маленькое, то другое –) экстремально большое, тк они связаны W + W = S. Мы также могли бы брать не минимум, а максимум и просто отрезать по нижней границе (M_obs >= ...), тут не принципиально: важно просто всегда брать какое то статично упорядоченное число, чтобы мы могли составить единвый критерий отвержения.

Так как у нас опять же это условие включает в себя и нижнюю и верхнюю границу, то можем просто написать на самом деле $p_{value} = 2P(W^+ \le T_{obs}) $ изза симметрии 

Получается, что критерий отвержения будет такой: $\quad W_{obs}^+ \le c_{\alpha}\quad или \quad W_{obs}^+ \ge S - c_{\alpha}, \quad $ где $c_{\alpha} = max\{c: P(M \le c) \le \alpha\} $\
(здесь S – просто сумма всех рангов)

Считать это мы можем как раз тем дп алгоритмом, который разбирали в 3.1.5



## 4. Перестановочный тест: 


Перестановочный тест в этой постановке задачи будет использовать идею очень похожую на тест Вилкоксона.

Мы примем похожую широкую нулевую гипотезу: 

$H_0: F_i = G_i \quad \forall i=1..n $.

и будем искать ей противоречия в данных.

**4.1.1** Сомнительная идея: возьмем похожий перестановочный тест и применим его. Будем случайно выбирать из множества в 2n наблюдений n наблюдений в одну группу и n наблюдений в другую группу (то есть смешаем "шум" и "тишину" в одну кучу и рассмотрим все возможные разделения на две равные группы). Соответственно, p-value будем считать как долю перестановок, в которых разница между медианами групп по модулю больше, чем в исходной перестановке.  
- Соответствует ли этот тест тому, какие перестановки разрешает наша нулевая гипотеза?
- Какие причины могут быть у низкого p-value в такой постановке теста?
  - На что среагирует тест, если распределения "шума" и распределения "тишины" различаются, но между всеми участниками внутри типа наблюдения они равны?
    - У всех одинаковое распределение "шума", и у всех другое одинаковое распределение "тишины"
  - Выполняется ли эта предпосылка в нашей задаче? Если нет, то на что еще может среагировать тест вместо искомого?
- Подходит ли этот тест для решения нашей прикладной задачи?

**4.1.2** Идею получше вам предстоит предложить самим:
- Опираясь на идею знаковой симметрии из теста Вилкоксона, предложите пространство "перестановок", разрешенное нашей нулевой гипотезой, и укажите его размерность.
  - "Пространство "перестановок"" = правило генерации "перестановок"
  - "Размерность" = число уникальных "перестановок"
  - *Подсказка:* для каждой перестановки вы можете запоминать некоторую характеристику полученного распределения разниц, чтбы получить ее перестановочное распределение.
- Сформулируйте двустороннюю альтернативную гипотезу и правило расчета точного p-value для неё по этим перестановкам.
- Прокомментируйте, будет ли здесь полезен метод оценивания p-value по Монте-Карло, или для n<100 можно обойтись точными расчетами?
- Какой размер B выборки "перестановок" необходимо взять, чтобы гарантировать, что стандартное отклонение p-value будет не больше 0.005? Сэмплировать нужно с возвращением или без возвращения?


**Подсказка: под "перестановками" здесь может подразумеваться другой комбинаторный объект; просто термин прижился**

---

4.1.1

1. Нет, потому что гипотеза говорит о том, что для участника i наблюдение в шуме и наблюдение в тишине можно поменять местами, потому что при H_0 они пришли из одного и того же распределения. Но она не говорит, что наблюдение участника i можно свободно поменять с наблюдением участника j. 
2. если две искусственно сформированные группы сильно различаются по выбранной статистике, например по медиане. Но проблема в том, что такой тест может реагировать не только на нужный эффект "шум против тишины". Он может реагировать на различия между двумя пулами наблюдений (X, y) без учета того, что X_i и Y_i связаны внутри одного участника.
3. Ну в таком случае у нас эта идея перестановки будет работать, тк мы действительно будем сравнивать различия F и G
4. У нас на данных она вряд ли будет выполнятся, тк у нас как минимум базовая реакция у всех людей разная. Он опять же сможет просто среагировать на различие пулов наблюдений, а не на сам парный эффект.
5. Скорее нет, потому то он игнорирует парную структуру, требует более сильную H_0, чем нам дана, тестирует вообще не то (при текущей H_0) и может давать сомнительные pvalue при неравности участников

4.1.2

Учитывая что у нас внутри одного участника распределения с шумом и без равны, мы можем как раз менять значения внутри участника. То есть у нас может быть две версии:
1. $(T_i, N_i)$
2. $(N_i, T_i)$

Тогда размерность этого пространства будет равна $2^n$ , тк у каждого участника есть два варианта: не менять ничего, поменять местами

Ну а дальше можем просто взять разность (в определенном порядке, например первый элемент минус второй) и смотреть на ее медиану. Тогда у нас при H_0 медиана будет равна 0, а при H_1 – не равна 0

Вообще для удобства записи лучше ввести новую св:  s = {-1, 1}. Тогда у нас по факту новые значения будут равны $s_i\cdot D_i $ , где D_i – "упорядоченная" разность. Тогда мы можем записывать результат перестановки как $(s_1\cdot D_1, ..., s_n\cdot D_n) $

Пусть $\quad T_{obs} = median(D_1, ..., D_n)\quad $ , а $\quad T_s = median(s_1\cdot D_1, ..., s_n\cdot D_n)$

Далее считаем эти статистики для всехъ $2^n$ комбинаций и получаем распределения медианы разниц при H_0

Дальше нам будет очень молезен метод Монте Карло, потому что если считать всн случаи, то это $2^n$ случаев, что даже при n=100 очень много. Поэтому берем просто B перестановок и считаем сумму перестановок, у которых $|T_{obs}| \ge |T_s|$ и потом получаем $\hat{p}_{value} = \frac{sum}{B} $

Если мы все таки хочтим считать точно, то нам нужно будет просто перебрать все варианты и также найти число перестановок у которых выполняется условие $|T_{obs}| \ge |T_s|$ и 
поделить на $2^n$


Теперь найдем нужный размер B для sd <= 0.005:\
$$D(\hat{p}) = D(\frac{1}{B}\sum I_i) = \frac{1}{B^2} D(\sum I_i) = \frac{1}{B^2}\cdot B \cdot p(1-p) = \frac{p(1-p)}{B} \implies SD(\hat{p}) = \sqrt{\frac{p(1-p)}{B}} $$

$$p(1-p) = p - p^2 \rightarrow \text{парабола}\implies \text{вершина будет в точке}\space 0.25 $$

Тогда мы требуем, чтобы:
$$\sqrt{\frac{0.25}{B}} \le 0.005 \implies \boxed{B \ge 10000 }$$